In [8]:
import sys
import glob
import yaml
import pickle
import os
import awkward as ak
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
plt.style.use("~/evanstyle.mplstyle")
sys.path.append("../")
import Utilities as Util

import DataReduction 
import CryoAsicAnalysis


In [ ]:
#change path!
#input_path = "/p/lustre1/nexouser/data/StanfordData/ChargeModule/LXe_Run1/Gamma_Data_Post_Surgery_7_15_24/"
input_path = "/Volumes/CODEDRIVE/Gamma_Data_7_16_24/prereduced/6g24pt_sig/"
#change path!
#output_path = "/p/lustre2/nexouser/data/StanfordData/angelico/LXe_Run1_Processed_Data/Gamma_Data_Post_Surgery_7_15_24/Processed_Data/"
#output_path = "../../../data/MockTileRun1/Gamma_Data_Post_Surgery_7_15_24/reduced/"
output_path = "/Volumes/CODEDRIVE/Gamma_Data_7_16_24/reduced/temp/"

input_files = glob.glob(input_path+"*.p")
try:
	print(input_files[0])
except:
	print("No files found in input directory")
	print("Does directory exist?: ", os.path.isdir(input_path))

In [10]:
config_path = "../config/gamma-post-surg-24.yml"
#either pass this directly to the class, or load it and modify with gain/pt settings corrected. 

#with open(config_path, 'r') as f:
#    config = yaml.safe_load(f)

#config["gain"] = 6 


In [ ]:
#initialize the DataReduction class
dr = DataReduction.DataReduction(config_path)
#load input data of many files, which combines the dataframes into one
for i, infile in enumerate(input_files):
	print("Reducing file {}".format(infile))
	dr.load_prereduced_data(infile)
	dr.basic_waveform_properties() #does basic properties of every channel's waveforms
	dr.identify_major_pulses() #uses a threshold discriminator to find major pulses and calculates their properties
	dr.process_clusters() #clusters pulses into events and measures properties
	dr.process_globals() #measures global properties of the event from the clusters
	dr.dictify_objects() #deletes Pulse and Cluster objects, turning them into dictionaries in the reduced df
	#dr.save_reduced_df(output_path, infile.split("/")[-1]) #saves the reduced df to a pickle file


In [ ]:

#combine the reduced files
print("Combining reduced files")
combined_df = pd.DataFrame()
for i, infile in enumerate(glob.glob(output_path+"*.p")):
	if("combined" in infile):
		continue
	df = pickle.load(open(infile, "rb"))[0]
	if(i == 0):
		combined_df = df
	else:
		combined_df = pd.concat([combined_df, df], ignore_index=True)

pickle.dump([combined_df], open(output_path+"combined.p", "wb"))